In [58]:

# in this notebook we will implement the remaining parts of the gpt model 
# we have already implemented the multihead attention mechanism
# we will need now to implement the feed forward network the layer normalization and the residual connections
# then finally stack them all together to form the transformer block
# we will also implement the GELU function (gaussian error linear unit) which is a smooth approximation of the ReLU function
# and is used as the activation function in the feed forward network
# its advantage is that it is differentiable and has a non-zero gradient for negative inputs 
# which helps with the vanishing gradient problem
# it tackles the dying relu problem by allowing negative inputs to have a small positive output
# except at approximately -0.75 


In [61]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import torch
from torch import nn

torch.manual_seed(42)

In [62]:
# a layer normalization layer is a type of normalization layer that normalizes the inputs before passing them the next layer
# this is done by subtracting the mean and dividing by the standard deviation of the inputs
# however we also multiply by learnable scale and add a learnable shift parameter to the normalized inputs 
# which will allow the model to learn optimal scale and shift if that is what will make the model perform better
# provides some kind of flexibility to the model to learn the optimal scale and shift parameters for the inputs

class LayerNorm(nn.Module):
    def __init__(self, din):
        super().__init__()

        self.gamma = nn.Parameter(torch.ones(din))
        self.beta = nn.Parameter(torch.zeros(din))

    def forward(self, X):
        mean = X.mean(dim = -1, keepdim = True)
        var = X.var(dim = -1, keepdim = True, unbiased = False)  # this prevents the bias towards the /n or /n-1 (bessels correction)
        std = torch.sqrt(var + 1e-5)  # add a small value to prevent division by zero

        norm_x = (X - mean) / std

        return self.gamma * norm_x + self.beta

In [63]:
# next up is implementing the gelu function
# and showing the difference between it and the relu one

# the GELU(X) = O(X) * X where 0 is the cummulitive distribution function of standard normal distribution 
# however the approximation of it is
# 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return 0.5 * X * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (X + 0.044715 * torch.pow(X, 3))))


In [64]:
# finally we can start implementing the feed forward network
# a fnn is just a network that takes in the context vectors and asks each "individual questions" enhancing
# each vector individually 
# by taking in the input vector then mapping it to a higher dimensional space to capture richer patterns then compressing it back

class FeedForwardNetwork(nn.Module):
    def __init__(self, din, expanding_factor = 4):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(din, expanding_factor * din),
            GELU(),
            nn.Linear(expanding_factor * din, din)
        )

    def forward(self, X):
        return self.layers(X)




In [65]:
# we can now finally start implementing the transformer architecture
# the transformer is designed to preserve the input dimensions
# this is a crucial part of it as it only takes in the input enhances them then outputs them in the same dimensions


# it consists of firstly a layernorm to prepare the inputs for the next transformation
# we also save the inputs prelayer norm as a shortcut connection
# then we pass them to the multihead attention
# enhancing them followed by a dropout to regularize the model to prevent overfitting
# then we apply a residual connection after the dropout 
# followed by phase 2 of the transformer which is the feed forward network again we first normalize the inputs
# then pass them to fnn followed by dropout then a final res connection

from src.multihead import MultiHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self, din, context_length, n_heads, expanding_factor, multihead_dropout, dropout):
        super().__init__()

        self.norm1 = LayerNorm(din)
        self.multiheadattention = MultiHeadAttention(din, din, n_heads, multihead_dropout, context_length)
        self.dropout = nn.Dropout(dropout)

        self.norm2 = LayerNorm(din)
        self.fnn = FeedForwardNetwork(din, expanding_factor)


    def forward(self, X):

        shortcut = X
        X = self.norm1(X)
        X = self.multiheadattention(X)
        X = self.dropout(X)

        X = X + shortcut

        shortcut = X

        X = self.norm2(X)
        X = self.fnn(X)
        X = self.dropout(X)
        X = X + shortcut

        return X


In [66]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

config = {
    'context_length': 256,
    'emb_dim': 512,
    'n_heads': 8,
    'mha_dropout': 0.1,
    'emb_dropout': 0.1,
    'dropout': 0.1,
    'expanding_factor': 4,
    'vocab_size': tokenizer.n_vocab,
    'n_layers': 8
}

In [69]:
class GPT8TModel(nn.Module):
    def __init__(self, emb_dim, vocab_size, context_length, n_layers, n_heads, emb_dropout, mha_dropout, dropout, expanding_factor = 4,
                 ):
        super().__init__()

        self.tok_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(context_length, emb_dim)

        self.embeds_dropout = nn.Dropout(emb_dropout)

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(emb_dim, context_length, n_heads, expanding_factor, mha_dropout, dropout) for _ in range(n_layers)]
        )

        self.final_layernorm = LayerNorm(emb_dim)

        self.output_projection = nn.Linear(emb_dim, vocab_size, bias = False)

        # we use weight tying where input weights == output weights to act as a sort of regularization + will offer 
        # a great computational efficiencty due to less number of weights
        self.output_projection.weight = self.tok_emb.weight

    def forward(self, X):
        n_batches, sequences = X.shape

        tok_embeds = self.tok_emb(X)
        pos_embeds = self.pos_emb(torch.arange(sequences, device = X.device))

        input_embeds = tok_embeds + pos_embeds

        # next up is applying the dropout
        input_embeds = self.embeds_dropout(input_embeds)

        # now we feed it to the trf blocks

        context_vectors = self.trf_blocks(input_embeds)

        context_vectors = self.final_layernorm(context_vectors)

        return self.output_projection(context_vectors)

In [70]:
model = GPT8TModel(config['emb_dim'], config['vocab_size'], config['context_length'], config['n_layers'], config['n_heads'],
                    config['emb_dropout'], config['mha_dropout'], config['dropout'])

In [71]:
# calculate model's size
import builtins
sum_parameters = builtins.sum((param.numel() for param in model.parameters()))

print(f"The model has {sum_parameters:,} total parameters")

# since each parameter is represented in float 32 bits
# then to calculate total bytes nparams * 32 // 4
models_size = (sum_parameters * 32 // 4) / (1024 * 1024)
print(f'The model is {models_size:.2f}MB')
print(f'Approximately {models_size / 1024:.2f}GB')


The model has 51,070,464 total parameters
The model is 389.64MB
Approximately 0.38GB


In [74]:
path_to_data = Path().cwd().parents[1] / 'data' / 'train_data.txt'

with open(path_to_data, 'r', encoding = 'utf-8') as f:
    raw_text = f.read()

print(raw_text[:1000])

Tahirah has been volunteering with Ethnic Minorities and Youth Support Team (EYST) Wales for over 5 years now, whilst contributing to the community and enjoying every minute of it. From play schemes to homework club, working with different community groups, she feels this has shaped her as a young individual.
In 2018, Tahirah became involved with Young, Migrant and Welsh (YMW), a project that focused on changing the perceptions of young individuals who live in Wales and are from a migrant community. Her contribution focused on females in sports, specifically representation and weightlifting.
Nominated as a Youth Ambassador for Wales by EYST Wales for the #iwill campaign Tahirah feels that she was able to reach a wider audience and raise more awareness, not only in the local community but nationally too. Whilst working with other Youth Ambassadors in Wales, Tahirah was able to contribute in creating material that reflected the diverse Wales.
Tahirah has also been involved in the judging

In [75]:
# now we use the train_loader instead 
from src.dataloader import create_dataloaderV0

dataloader = create_dataloaderV0(raw_text[:100_000], max_window_length=6, stride=6, batch_size=1,
                                shuffle=True, drop_last=True, tokenizer= tokenizer, num_workers=0)

In [78]:
data_iter = iter(dataloader)

def generate_text_tokIDs(model, tokenIDs, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        context_idx = tokenIDs[:, -context_size:]

        with torch.no_grad():
            logits = model(context_idx)

        logits = logits[:, -1, :]   # take only the last token
        probas = torch.softmax(logits, dim = -1)
        next_tokID = probas.argmax(dim = -1, keepdim=True)
        if (next_tokID == tokenizer.eot_token).any():
            break
        tokenIDs = torch.cat((tokenIDs, next_tokID), dim = 1)

    return tokenIDs

In [79]:
def generate_and_print_sample(model, tokenizer, device, start_context, context_size):
    model.eval()
    context_size = context_size
    encoded = tokenizer.encode(start_context, allowed_special = {'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0).to(device)

    with torch.no_grad():
        token_ids = generate_text_tokIDs(model = model, tokenIDs=encoded_tensor, max_new_tokens=50, context_size = context_size)

    decoded_text = tokenizer.decode(token_ids.squeeze(0).tolist())

    print(decoded_text.replace('\n', ' '))
    model.train()

In [80]:
model.eval()
predictions = generate_text_tokIDs(model = model, tokenIDs=next(data_iter)[1], max_new_tokens= 5, context_size=config['context_length'])
predictions.shape

torch.Size([1, 11])

In [82]:
preds = tokenizer.decode(predictions[0].squeeze(0).tolist())
preds

' beyond the reach of many......'